# Predictive Analytics: NN vs. SVM

This notebook compares the final neural-network and support-vector-regression pipelines on every dataset for which both models provide test predictions.

The comparison is deliberately rebuilt from row-level predictions instead of loading previously aggregated NN or SVM metric tables. Predictions are joined by timestamp and spatial key, and the corresponding test-baseline artifact supplies one canonical `y_true` column for both models. This prevents small target differences between separately generated artifacts from affecting the comparison.

The comparison represents the complete fitted pipelines, not an isolated architecture experiment: NN and SVM may differ in training sample size, feature representation and model-selection objective.

## Setup

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from run_config import MODELS_DIR, PATHS, RUN_MODE

In [ ]:
NN_DIR = MODELS_DIR / "nn_final"
SVM_DIR = MODELS_DIR.parent / "svm"
BASELINE_DIR = PATHS.train_test_dir / "baseline_predictions"
OUTPUT_DIR = MODELS_DIR / "svm_nn_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_METRICS_PATH = OUTPUT_DIR / "svm_nn_model_metrics.parquet"

NN_VERSION = "final_nn_v1"
REFERENCE_BASELINE_COLS = {
    "zero": "prediction_zero",
    "global_median": "prediction_global_median",
    "spatial_time_weekday_mean": "prediction_spatial_time_weekday_mean",
}
DEMAND_THRESHOLD = 0.5
BALANCED_MAE_ALPHA = 0.5
PEAK_QUANTILE = 0.90

for directory in (NN_DIR, SVM_DIR, BASELINE_DIR):
    if not directory.exists():
        raise FileNotFoundError(f"Required artifact directory does not exist: {directory}")

print(f"NN predictions:       {NN_DIR}")
print(f"SVM predictions:      {SVM_DIR}")
print(f"Canonical test truth: {BASELINE_DIR}")

## Discover and pair all available datasets

No dataset list is hard-coded. New configurations are included automatically as soon as matching NN, SVM and baseline prediction artifacts exist. Configurations available for only one model are reported rather than silently discarded.

In [ ]:
SVM_FILE_PATTERN = re.compile(
    r"^model_(CENSUS_TRACTS|COMMUNITY_AREAS|HEXAGON_(\d+))_(1H|4H|24H)\.csv$"
)
NN_FILE_PATTERN = re.compile(
    rf"^{NN_VERSION}_(.+)_(1h|4h|24h)_test_predictions\.parquet$"
)


def spatial_key_for(dataset: str) -> str:
    if dataset.startswith("hexagon_h3r"):
        return "h3_cell"
    if dataset.startswith("census_tracts"):
        return "census_tract"
    if dataset.startswith("community_areas"):
        return "community_area"
    raise ValueError(f"Unknown spatial dataset tag: {dataset}")


def svm_dataset_tag(match: re.Match) -> str:
    spatial_tag, h3_resolution, time_unit = match.groups()
    if spatial_tag.startswith("HEXAGON_"):
        return f"hexagon_h3r{h3_resolution}_{time_unit.lower()}"
    return f"{spatial_tag.lower()}_{time_unit.lower()}"


svm_catalog = {}
for path in sorted(SVM_DIR.glob("model_*.csv")):
    match = SVM_FILE_PATTERN.match(path.name)
    if match:
        svm_catalog[svm_dataset_tag(match)] = path

nn_catalog = {}
for path in sorted(NN_DIR.glob(f"{NN_VERSION}_*_test_predictions.parquet")):
    match = NN_FILE_PATTERN.match(path.name)
    if match:
        spatial_tag, time_unit = match.groups()
        nn_catalog[f"{spatial_tag}_{time_unit}"] = path

common_datasets = sorted(set(nn_catalog) & set(svm_catalog))
nn_only_datasets = sorted(set(nn_catalog) - set(svm_catalog))
svm_only_datasets = sorted(set(svm_catalog) - set(nn_catalog))

if not common_datasets:
    raise ValueError("No matching NN and SVM test-prediction artifacts were found.")

coverage = pl.DataFrame({
    "coverage": ["paired", "NN only", "SVM only"],
    "n_datasets": [len(common_datasets), len(nn_only_datasets), len(svm_only_datasets)],
    "datasets": [
        ", ".join(common_datasets),
        ", ".join(nn_only_datasets) or "—",
        ", ".join(svm_only_datasets) or "—",
    ],
})
coverage

## Load, align and audit predictions

For every paired dataset, both prediction files must contain exactly the same unique timestamp–location keys as the baseline test artifact. Stored NN and SVM targets are retained only for auditing; every metric below uses the baseline artifact's `y_true` as the shared target.

In [ ]:
def normalized_time(frame: pl.DataFrame) -> pl.DataFrame:
    return frame.with_columns(pl.col("datetime_hour").cast(pl.Datetime("us")))


def assert_unique_keys(frame: pl.DataFrame, keys: list[str], label: str) -> None:
    n_unique = frame.select(pl.struct(keys).n_unique()).item()
    if n_unique != frame.height:
        raise ValueError(
            f"{label} contains {frame.height - n_unique:,} duplicate timestamp-location keys."
        )


def load_aligned_dataset(dataset: str) -> tuple[pl.DataFrame, dict]:
    spatial_key = spatial_key_for(dataset)
    join_keys = ["datetime_hour", spatial_key]
    baseline_path = BASELINE_DIR / f"{dataset}_test_baselines.parquet"
    if not baseline_path.exists():
        raise FileNotFoundError(f"Missing canonical baseline predictions: {baseline_path}")

    baseline = normalized_time(
        pl.read_parquet(baseline_path).select(
            join_keys + ["y_true", *REFERENCE_BASELINE_COLS.values()]
        )
    )
    nn = normalized_time(
        pl.read_parquet(nn_catalog[dataset]).select(
            join_keys
            + [
                pl.col("y_true").alias("y_true_nn_artifact"),
                pl.col("y_pred").alias("nn_prediction"),
            ]
        )
    )
    svm = normalized_time(
        pl.read_csv(svm_catalog[dataset], try_parse_dates=True)
        .rename({"date": "datetime_hour"})
        .select(
            join_keys
            + [
                pl.col("y_test").alias("y_true_svm_artifact"),
                pl.col("y_pred").alias("svm_prediction"),
            ]
        )
    )

    for label, frame in (("baseline", baseline), ("NN", nn), ("SVM", svm)):
        assert_unique_keys(frame, join_keys, f"{dataset} {label}")

    nn_missing = baseline.join(nn, on=join_keys, how="anti").height
    nn_extra = nn.join(baseline, on=join_keys, how="anti").height
    svm_missing = baseline.join(svm, on=join_keys, how="anti").height
    svm_extra = svm.join(baseline, on=join_keys, how="anti").height
    if any((nn_missing, nn_extra, svm_missing, svm_extra)):
        raise ValueError(
            f"Key mismatch for {dataset}: NN missing/extra={nn_missing}/{nn_extra}, "
            f"SVM missing/extra={svm_missing}/{svm_extra}."
        )

    aligned = baseline.join(nn, on=join_keys, how="inner").join(
        svm, on=join_keys, how="inner"
    )
    required_values = [
        "y_true", *REFERENCE_BASELINE_COLS.values(), "nn_prediction", "svm_prediction"
    ]
    null_count = aligned.select(
        pl.sum_horizontal([pl.col(column).null_count() for column in required_values])
        .alias("null_count")
    ).item()
    if null_count:
        raise ValueError(f"{dataset} contains {null_count:,} null comparison values.")

    nn_target_error = (
        pl.col("y_true") - pl.col("y_true_nn_artifact")
    ).abs()
    svm_target_error = (
        pl.col("y_true") - pl.col("y_true_svm_artifact")
    ).abs()
    audit = {
        "dataset": dataset,
        "n_rows": aligned.height,
        "nn_target_mismatches": aligned.filter(nn_target_error > 0).height,
        "nn_max_target_difference": aligned.select(nn_target_error.max()).item(),
        "svm_target_mismatches": aligned.filter(svm_target_error > 0).height,
        "svm_max_target_difference": aligned.select(svm_target_error.max()).item(),
        "nn_negative_predictions": aligned.filter(pl.col("nn_prediction") < 0).height,
        "svm_negative_predictions": aligned.filter(pl.col("svm_prediction") < 0).height,
    }
    return aligned, audit

In [ ]:
aligned_predictions = {}
audit_rows = []

for dataset in common_datasets:
    aligned, audit = load_aligned_dataset(dataset)
    aligned_predictions[dataset] = aligned
    audit_rows.append(audit)

alignment_audit = pl.DataFrame(audit_rows).sort("dataset")
alignment_audit

Target mismatch counts are diagnostic only. They document differences in separately stored target columns; the metric calculation always uses the common baseline `y_true`.

## Compute comparable high-level metrics

- **Demand MAE** is the MAE for observations with `y_true > 0`.
- **Zero MAE** is the MAE for observations with `y_true = 0`.
- **Balanced MAE** weights Zero MAE and Demand MAE equally.
- **Peak MAE** uses the upper 10% of positive target observations within each dataset.
- Precision and recall treat predictions of at least 0.5 trips as positive demand.
- Three MAE skill scores are calculated on the complete test set. Their references are the zero, global-median and spatial × time × weekday mean baselines. Every score follows `1 - model MAE / baseline MAE`.
- No separate skill scores are calculated for zero-demand, positive-demand, balanced or peak subsets.

In [ ]:
def safe_mean(values: np.ndarray) -> float:
    return float(values.mean()) if values.size else float("nan")


def safe_skill(model_error: float, baseline_error: float) -> float:
    return (
        float(1.0 - model_error / baseline_error)
        if np.isfinite(baseline_error) and baseline_error > 0
        else float("nan")
    )


def evaluate_model(
    dataset: str,
    frame: pl.DataFrame,
    *,
    model: str,
    prediction_col: str,
) -> dict:
    y_true = frame["y_true"].to_numpy().astype(float)
    y_pred = frame[prediction_col].to_numpy().astype(float)
    baseline_predictions = {
        name: frame[column].to_numpy().astype(float)
        for name, column in REFERENCE_BASELINE_COLS.items()
    }

    finite = np.isfinite(y_true) & np.isfinite(y_pred)
    for baseline_pred in baseline_predictions.values():
        finite &= np.isfinite(baseline_pred)
    y_true, y_pred = y_true[finite], y_pred[finite]
    baseline_predictions = {
        name: prediction[finite]
        for name, prediction in baseline_predictions.items()
    }
    if y_true.size == 0:
        raise ValueError(f"No finite observations available for {dataset} {model}.")

    error = y_true - y_pred
    abs_error = np.abs(error)
    squared_error = error**2
    zero_mask = y_true == 0
    demand_mask = y_true > 0

    if not zero_mask.any() or not demand_mask.any():
        raise ValueError(f"{dataset} must contain both zero and positive demand.")

    peak_threshold = float(np.quantile(y_true[demand_mask], PEAK_QUANTILE))
    peak_mask = y_true >= peak_threshold

    mae = safe_mean(abs_error)
    zero_mae = safe_mean(abs_error[zero_mask])
    demand_mae = safe_mean(abs_error[demand_mask])
    balanced_mae = (
        (1.0 - BALANCED_MAE_ALPHA) * zero_mae
        + BALANCED_MAE_ALPHA * demand_mae
    )
    peak_mae = safe_mean(abs_error[peak_mask])
    demand_bucket_masks = {
        "0": y_true == 0,
        "1": y_true == 1,
        "2_3": (y_true >= 2) & (y_true <= 3),
        "4_5": (y_true >= 4) & (y_true <= 5),
        "6_10": (y_true >= 6) & (y_true <= 10),
        "11_20": (y_true >= 11) & (y_true <= 20),
        "gt_20": y_true > 20,
    }
    demand_bucket_mae = {
        f"demand_bucket_mae_{bucket}": safe_mean(abs_error[mask])
        for bucket, mask in demand_bucket_masks.items()
    }
    demand_bucket_mean_y_true = {
        f"demand_bucket_mean_y_true_{bucket}": safe_mean(y_true[mask])
        for bucket, mask in demand_bucket_masks.items()
    }

    full_test_baseline_mae = {
        f"baseline_mae_{name}": safe_mean(np.abs(y_true - baseline_pred))
        for name, baseline_pred in baseline_predictions.items()
    }
    full_test_skill_scores = {
        f"mae_skill_vs_{name}": safe_skill(
            mae, full_test_baseline_mae[f"baseline_mae_{name}"]
        )
        for name in baseline_predictions
    }

    actual_positive = demand_mask
    predicted_positive = y_pred >= DEMAND_THRESHOLD
    true_positive = np.logical_and(actual_positive, predicted_positive).sum()
    precision = (
        float(true_positive / predicted_positive.sum())
        if predicted_positive.any()
        else float("nan")
    )
    recall = float(true_positive / actual_positive.sum())

    ss_res = squared_error.sum()
    ss_total = ((y_true - y_true.mean()) ** 2).sum()
    r2 = float(1.0 - ss_res / ss_total) if ss_total > 0 else float("nan")

    return {
        "dataset": dataset,
        "model": model,
        "n_samples": y_true.size,
        "mae": mae,
        "rmse": float(np.sqrt(squared_error.mean())),
        "r2": r2,
        "zero_mae": zero_mae,
        "demand_mae": demand_mae,
        "balanced_mae": balanced_mae,
        "peak_threshold": peak_threshold,
        "peak_mae": peak_mae,
        "actual_positive_share": float(actual_positive.mean()),
        "predicted_positive_share": float(predicted_positive.mean()),
        "demand_precision": precision,
        "demand_recall": recall,
        **demand_bucket_mae,
        **demand_bucket_mean_y_true,
        **full_test_baseline_mae,
        **full_test_skill_scores,
    }

In [ ]:
metric_rows = []
for dataset, frame in aligned_predictions.items():
    metric_rows.append(
        evaluate_model(dataset, frame, model="NN", prediction_col="nn_prediction")
    )
    metric_rows.append(
        evaluate_model(dataset, frame, model="SVM", prediction_col="svm_prediction")
    )

model_metrics = pl.DataFrame(metric_rows).sort(["dataset", "model"])
model_metrics.write_parquet(MODEL_METRICS_PATH)
print(f"Saved model metrics to {MODEL_METRICS_PATH}")
model_metrics

## High-level NN vs. SVM comparison for every paired dataset

In [ ]:
CORE_METRICS = [
    "r2",
    "mae",
    "rmse",
    "demand_mae",
    "zero_mae",
    "balanced_mae",
    "peak_mae",
    "demand_precision",
    "demand_recall",
    "mae_skill_vs_zero",
    "mae_skill_vs_global_median",
    "mae_skill_vs_spatial_time_weekday_mean",
]

high_level_long = model_metrics.unpivot(
    index=["dataset", "model"],
    on=CORE_METRICS,
    variable_name="metric",
    value_name="value",
)

high_level_comparison = (
    high_level_long
    .pivot(on="model", index=["dataset", "metric"], values="value")
    .with_columns((pl.col("SVM") - pl.col("NN")).alias("svm_minus_nn"))
    .sort(["dataset", "metric"])
)

for dataset in common_datasets:
    print(f"\n{'=' * 72}\n{dataset}\n{'=' * 72}")
    display(
        high_level_comparison
        .filter(pl.col("dataset") == dataset)
        .select("metric", "NN", "SVM", "svm_minus_nn")
        .with_columns(pl.col("NN", "SVM", "svm_minus_nn").round(4))
    )

For error metrics, a negative `svm_minus_nn` favors SVM; for R², precision, recall and skill scores, a positive value favors SVM.

In [ ]:
HIGHER_IS_BETTER = {
    "r2",
    "demand_precision",
    "demand_recall",
    "mae_skill_vs_zero",
    "mae_skill_vs_global_median",
    "mae_skill_vs_spatial_time_weekday_mean",
}

winner_summary = high_level_comparison.with_columns(
    pl.when(pl.col("NN").is_nan() | pl.col("SVM").is_nan())
    .then(pl.lit("not comparable"))
    .when(pl.col("NN") == pl.col("SVM"))
    .then(pl.lit("tie"))
    .when(
        (pl.col("metric").is_in(HIGHER_IS_BETTER) & (pl.col("NN") > pl.col("SVM")))
        | (~pl.col("metric").is_in(HIGHER_IS_BETTER) & (pl.col("NN") < pl.col("SVM")))
    )
    .then(pl.lit("NN"))
    .otherwise(pl.lit("SVM"))
    .alias("winner")
)

winner_counts = (
    winner_summary
    .group_by(["metric", "winner"])
    .len()
    .pivot(on="winner", index="metric", values="len")
    .fill_null(0)
    .sort("metric")
)
winner_counts

## Visual overview

In [ ]:
PLOT_METRICS = [
    ("mae", "MAE"),
    ("demand_mae", "Demand MAE"),
    ("zero_mae", "Zero MAE"),
    ("r2", "R²"),
    ("demand_precision", "Demand precision"),
    ("demand_recall", "Demand recall"),
]

figure, axes = plt.subplots(3, 2, figsize=(16, 16))
x = np.arange(len(common_datasets))
width = 0.38

for axis, (metric, title) in zip(axes.flat, PLOT_METRICS):
    metric_frame = high_level_comparison.filter(pl.col("metric") == metric).sort("dataset")
    axis.bar(x - width / 2, metric_frame["NN"].to_numpy(), width, label="NN")
    axis.bar(x + width / 2, metric_frame["SVM"].to_numpy(), width, label="SVM")
    axis.set_title(title)
    axis.set_xticks(x)
    axis.set_xticklabels(metric_frame["dataset"].to_list(), rotation=45, ha="right")
    axis.grid(axis="y", alpha=0.3)
    axis.legend()

figure.suptitle("Final NN and SVM pipelines on the same canonical test targets", y=1.01)
figure.tight_layout()
plt.show()

## MAE skill score comparison

This figure compares the two most relevant full-test skill scores. Positive values indicate an improvement over the respective baseline; negative values indicate that the baseline has the lower MAE.

In [ ]:
from matplotlib.ticker import PercentFormatter

TIME_ORDER = {"1h": 0, "4h": 1, "24h": 2}
SPATIAL_ORDER = {
    "census_tracts": 0,
    "community_areas": 1,
    "hexagon_h3r7": 2,
}


def dataset_sort_key(dataset: str) -> tuple[int, int]:
    spatial, time_unit = dataset.rsplit("_", 1)
    return SPATIAL_ORDER.get(spatial, 99), TIME_ORDER.get(time_unit, 99)


def dataset_display_name(dataset: str) -> str:
    spatial, time_unit = dataset.rsplit("_", 1)
    spatial_name = {
        "census_tracts": "Census tracts",
        "community_areas": "Community areas",
        "hexagon_h3r7": "H3 resolution 7",
    }.get(spatial, spatial.replace("_", " ").title())
    return f"{spatial_name} · {time_unit.upper()}"


skill_plot_datasets = sorted(common_datasets, key=dataset_sort_key)
skill_plot_specs = [
    ("mae_skill_vs_zero", "MAE skill vs. zero-demand baseline"),
    (
        "mae_skill_vs_spatial_time_weekday_mean",
        "MAE skill vs. spatial × time × weekday baseline",
    ),
]
model_colors = {"NN": "#1f77b4", "SVM": "#ff7f0e"}

figure, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
x = np.arange(len(skill_plot_datasets))
width = 0.38

for axis, (metric, title) in zip(axes, skill_plot_specs):
    metric_frame = (
        model_metrics
        .select("dataset", "model", metric)
        .pivot(on="model", index="dataset", values=metric)
    )
    values_by_dataset = {
        row["dataset"]: row for row in metric_frame.iter_rows(named=True)
    }

    for offset, model in ((-width / 2, "NN"), (width / 2, "SVM")):
        values = [values_by_dataset[dataset][model] for dataset in skill_plot_datasets]
        axis.bar(
            x + offset,
            values,
            width,
            label=model,
            color=model_colors[model],
        )

    axis.axhline(0, color="black", linewidth=1)
    axis.set_title(title, loc="left")
    axis.set_ylabel("MAE skill score")
    axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    axis.grid(axis="y", alpha=0.25)
    axis.legend(frameon=False, ncols=2)

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(
    [dataset_display_name(dataset) for dataset in skill_plot_datasets],
    rotation=35,
    ha="right",
)
figure.suptitle(
    "MAE skill scores by model and dataset",
    fontsize=16,
)
figure.tight_layout()
plt.show()

## Compact machine-readable result tables

`model_metrics` contains one row per dataset and model. `high_level_comparison` contains paired NN and SVM values plus their difference. `alignment_audit` documents key and target consistency. These in-memory tables can be reused by later report or diagnostic sections without loading the legacy `results_svr.csv` summary.